# Vietnamese OCR: Text Detection & Recognition

This notebook implements a complete OCR pipeline for Vietnamese text, combining two complementary approaches:

## OCR Pipeline

1. **Text Detection** - Locating text regions in images (bounding boxes)
2. **Text Recognition** - Converting detected text regions into readable strings

## Models Used

| Model | Role | Strengths |
|-------|------|----------|
| **PaddleOCR** | Detection + Recognition | Fast, all-in-one pipeline, supports Vietnamese |
| **VietOCR** | Recognition only | Purpose-built for Vietnamese, superior diacritic handling |

## Vietnamese-Specific Challenges

Vietnamese text presents unique OCR challenges:
- **Diacritical marks (dấu)**: Vietnamese uses 6 tones marked by diacritics (sắc, huyền, hỏi, ngã, nặng, ngang)
- **Extended Latin characters**: ă, â, đ, ê, ô, ơ, ư and their tonal combinations
- **Unicode complexity**: A single character like `ệ` combines base `e` + circumflex `ê` + dot below `ệ`
- **High visual similarity**: Characters like `a/ă/â`, `o/ô/ơ`, `u/ư` differ by small marks that are easily missed

**Dataset**: [trongnguyen04/vietnamese-ocr](https://www.kaggle.com/datasets/trongnguyen04/vietnamese-ocr) (~1GB)

---
## 1. Install Dependencies & Import Libraries

In [ ]:
# Install required packages
!pip install -q paddlepaddle paddleocr vietocr Pillow matplotlib opencv-python-headless

In [ ]:
import os
import glob
import json
import warnings
warnings.filterwarnings('ignore')

import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from paddleocr import PaddleOCR
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

# Display settings
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['figure.dpi'] = 100

print("All imports successful.")

---
## 2. Download & Locate Dataset

In [ ]:
# Locate or download the dataset
DATA_DIR = "/kaggle/input/vietnamese-ocr"

if not os.path.exists(DATA_DIR) or not os.listdir(DATA_DIR):
    print("Dataset not found in /kaggle/input. Downloading...")
    os.makedirs("/kaggle/working/data", exist_ok=True)
    ret = os.system("kaggle datasets download -d trongnguyen04/vietnamese-ocr -p /kaggle/working/data --unzip")
    if ret != 0:
        print("WARNING: kaggle CLI download failed. Trying opendatasets...")
        os.system("pip install -q opendatasets")
        import opendatasets as od
        od.download("https://www.kaggle.com/datasets/trongnguyen04/vietnamese-ocr",
                    data_dir="/kaggle/working/data")
    DATA_DIR = "/kaggle/working/data"
    # Navigate into subdirectory if the unzip created one
    subdirs = [d for d in os.listdir(DATA_DIR)
               if os.path.isdir(os.path.join(DATA_DIR, d))]
    if len(subdirs) == 1:
        DATA_DIR = os.path.join(DATA_DIR, subdirs[0])
else:
    print(f"Dataset found at {DATA_DIR}")

print(f"\nUsing DATA_DIR: {DATA_DIR}")
print(f"\nContents (first 15 items):")
for item in sorted(os.listdir(DATA_DIR))[:15]:
    full = os.path.join(DATA_DIR, item)
    kind = "[DIR]" if os.path.isdir(full) else f"[FILE {os.path.getsize(full)//1024}KB]"
    print(f"  {kind} {item}")

---
## 3. Explore the Dataset

In [ ]:
def find_files(root, extensions):
    """Recursively find files with given extensions."""
    files = []
    for ext in extensions:
        files.extend(glob.glob(os.path.join(root, '**', f'*{ext}'), recursive=True))
    return sorted(files)

# Find image and label files
image_files = find_files(DATA_DIR, ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'])
label_files = find_files(DATA_DIR, ['.txt', '.csv', '.json'])

print(f"Total image files: {len(image_files)}")
print(f"Total label/text files: {len(label_files)}")

# Show image file distribution by directory
dir_counts = {}
for f in image_files:
    d = os.path.dirname(f).replace(DATA_DIR, '').strip('/')
    d = d if d else '(root)'
    dir_counts[d] = dir_counts.get(d, 0) + 1

print("\nImages by directory:")
for d, c in sorted(dir_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {d}: {c} images")

In [ ]:
# Show sample images
sample_images = image_files[:8] if len(image_files) >= 8 else image_files

n = len(sample_images)
cols = min(4, n)
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
if rows == 1 and cols == 1:
    axes = np.array([axes])
axes = np.array(axes).flatten()

for i, ax in enumerate(axes):
    if i < n:
        img = Image.open(sample_images[i]).convert('RGB')
        ax.imshow(img)
        fname = os.path.basename(sample_images[i])
        ax.set_title(f"{fname}\n{img.size[0]}x{img.size[1]}", fontsize=9)
    ax.axis('off')

plt.suptitle('Sample Images from Dataset', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Image size statistics
print("\nImage size statistics (sampling up to 200 images):")
widths, heights = [], []
for f in image_files[:200]:
    try:
        img = Image.open(f)
        widths.append(img.size[0])
        heights.append(img.size[1])
    except Exception:
        pass

if widths:
    print(f"  Width  - min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}")
    print(f"  Height - min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}")

In [ ]:
# Examine label files if available
if label_files:
    print(f"Found {len(label_files)} label/text files.\n")
    for lf in label_files[:5]:
        print(f"--- {os.path.basename(lf)} ---")
        try:
            with open(lf, 'r', encoding='utf-8') as f:
                lines = f.readlines()[:5]
            for line in lines:
                print(f"  {line.rstrip()}")
        except Exception as e:
            print(f"  [Error reading: {e}]")
        print()
else:
    print("No separate label files found. Will rely on OCR output only.")

---
## 4. PaddleOCR - Full Pipeline (Detection + Recognition)

In [ ]:
# Initialize PaddleOCR with Vietnamese language support
paddle_ocr = PaddleOCR(
    lang='vi',           # Vietnamese
    use_angle_cls=True,  # Detect text orientation
    show_log=False,
    use_gpu=True         # Will fallback to CPU if no GPU available
)

print("PaddleOCR initialized (lang=vi).")

In [ ]:
def run_paddleocr(image_path):
    """
    Run PaddleOCR on a single image.
    Returns list of (bbox, text, confidence) tuples.
    """
    result = paddle_ocr.ocr(image_path, cls=True)
    detections = []
    if result and result[0]:
        for line in result[0]:
            bbox = line[0]          # [[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
            text = line[1][0]       # recognized text
            conf = line[1][1]       # confidence score
            detections.append((bbox, text, conf))
    return detections


def visualize_paddleocr(image_path, detections, title=None):
    """
    Draw PaddleOCR results: bounding boxes + text labels on the image.
    """
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

    # Left: image with bounding boxes
    ax1.imshow(img)
    colors = plt.cm.Set2(np.linspace(0, 1, max(len(detections), 1)))
    for idx, (bbox, text, conf) in enumerate(detections):
        pts = np.array(bbox)
        polygon = patches.Polygon(pts, linewidth=2, edgecolor=colors[idx % len(colors)],
                                  facecolor='none')
        ax1.add_patch(polygon)
        ax1.text(pts[0][0], pts[0][1] - 5, f"{idx+1}",
                 fontsize=8, color='white',
                 bbox=dict(boxstyle='round,pad=0.2', facecolor=colors[idx % len(colors)], alpha=0.8))
    ax1.set_title('Detected Text Regions', fontsize=12)
    ax1.axis('off')

    # Right: text listing
    ax2.axis('off')
    text_content = "Detected Text Lines:\n" + "=" * 40 + "\n\n"
    for idx, (bbox, text, conf) in enumerate(detections):
        text_content += f"{idx+1:2d}. [{conf:.3f}] {text}\n"
    if not detections:
        text_content += "(No text detected)"
    ax2.text(0.05, 0.95, text_content, transform=ax2.transAxes,
             fontsize=10, verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    ax2.set_title('Recognition Results', fontsize=12)

    suptitle = title or os.path.basename(image_path)
    plt.suptitle(f'PaddleOCR: {suptitle}', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Run PaddleOCR on sample images
NUM_SAMPLES = min(3, len(image_files))
paddle_results = {}

for img_path in image_files[:NUM_SAMPLES]:
    print(f"\nProcessing: {os.path.basename(img_path)}")
    detections = run_paddleocr(img_path)
    paddle_results[img_path] = detections
    print(f"  Found {len(detections)} text regions")
    visualize_paddleocr(img_path, detections)

---
## 5. VietOCR - Vietnamese-Specific Recognition

In [ ]:
# Initialize VietOCR with pretrained vgg_transformer model
vietocr_config = Cfg.load_config_from_name('vgg_transformer')
vietocr_config['cnn']['pretrained'] = True
vietocr_config['device'] = 'cuda:0'  # Will be overridden below if no GPU

import torch
if not torch.cuda.is_available():
    vietocr_config['device'] = 'cpu'
    print("No GPU available, using CPU for VietOCR.")
else:
    print("Using GPU for VietOCR.")

vietocr_predictor = Predictor(vietocr_config)
print("VietOCR initialized (vgg_transformer).")

In [ ]:
def crop_text_region(image_path, bbox):
    """
    Crop a text region from an image using a quadrilateral bounding box.
    Uses perspective transform to get a clean rectangular crop.
    """
    img = cv2.imread(image_path)
    pts = np.array(bbox, dtype=np.float32)

    # Compute width and height of the target rectangle
    width = int(max(
        np.linalg.norm(pts[0] - pts[1]),
        np.linalg.norm(pts[2] - pts[3])
    ))
    height = int(max(
        np.linalg.norm(pts[0] - pts[3]),
        np.linalg.norm(pts[1] - pts[2])
    ))

    if width < 2 or height < 2:
        return None

    dst_pts = np.array([
        [0, 0], [width, 0], [width, height], [0, height]
    ], dtype=np.float32)

    M = cv2.getPerspectiveTransform(pts, dst_pts)
    cropped = cv2.warpPerspective(img, M, (width, height))
    cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
    return Image.fromarray(cropped_rgb)


def run_vietocr_on_crops(image_path, detections):
    """
    Run VietOCR on cropped text regions from PaddleOCR detections.
    Returns list of (bbox, vietocr_text, paddle_text, paddle_conf).
    """
    results = []
    for bbox, paddle_text, paddle_conf in detections:
        crop = crop_text_region(image_path, bbox)
        if crop is not None:
            vietocr_text = vietocr_predictor.predict(crop)
            results.append((bbox, vietocr_text, paddle_text, paddle_conf))
        else:
            results.append((bbox, '[crop failed]', paddle_text, paddle_conf))
    return results

In [ ]:
# Compare PaddleOCR vs VietOCR on sample images
comparison_results = {}

for img_path in image_files[:NUM_SAMPLES]:
    detections = paddle_results.get(img_path, [])
    if not detections:
        continue

    print(f"\n{'='*70}")
    print(f"Image: {os.path.basename(img_path)}")
    print(f"{'='*70}")

    results = run_vietocr_on_crops(img_path, detections)
    comparison_results[img_path] = results

    print(f"{'#':>3} {'Conf':>6}  {'PaddleOCR':<35} {'VietOCR':<35}  {'Match?'}")
    print(f"{'-'*3} {'-'*6}  {'-'*35} {'-'*35}  {'-'*6}")
    for idx, (bbox, vietocr_text, paddle_text, conf) in enumerate(results):
        match = 'YES' if paddle_text.strip() == vietocr_text.strip() else 'no'
        p_display = paddle_text[:33] + '..' if len(paddle_text) > 35 else paddle_text
        v_display = vietocr_text[:33] + '..' if len(vietocr_text) > 35 else vietocr_text
        print(f"{idx+1:3d} {conf:6.3f}  {p_display:<35} {v_display:<35}  {match}")

    # Show cropped regions for visual inspection
    n_show = min(6, len(results))
    if n_show > 0:
        fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 2))
        if n_show == 1:
            axes = [axes]
        for i in range(n_show):
            crop = crop_text_region(img_path, results[i][0])
            if crop:
                axes[i].imshow(crop)
                axes[i].set_title(f"V: {results[i][1][:20]}", fontsize=8)
            axes[i].axis('off')
        plt.suptitle('Cropped Text Regions + VietOCR Output', fontsize=11)
        plt.tight_layout()
        plt.show()

---
## 6. Batch Processing

In [ ]:
# Process a larger batch of images
BATCH_SIZE = min(20, len(image_files))
batch_images = image_files[:BATCH_SIZE]

all_results = []

print(f"Processing {BATCH_SIZE} images...\n")
for i, img_path in enumerate(batch_images):
    try:
        # PaddleOCR detection + recognition
        detections = run_paddleocr(img_path)

        # VietOCR recognition on detected regions
        for bbox, paddle_text, paddle_conf in detections:
            crop = crop_text_region(img_path, bbox)
            vietocr_text = ''
            if crop is not None:
                try:
                    vietocr_text = vietocr_predictor.predict(crop)
                except Exception:
                    vietocr_text = '[error]'

            all_results.append({
                'image': os.path.basename(img_path),
                'paddle_text': paddle_text,
                'paddle_conf': round(paddle_conf, 4),
                'vietocr_text': vietocr_text
            })

        if (i + 1) % 5 == 0:
            print(f"  Processed {i+1}/{BATCH_SIZE} images ({len(all_results)} text regions so far)")

    except Exception as e:
        print(f"  Error processing {os.path.basename(img_path)}: {e}")

print(f"\nDone! Total text regions extracted: {len(all_results)}")

In [ ]:
# Display results as a table
print(f"{'Image':<25} {'Conf':>6}  {'PaddleOCR':<40} {'VietOCR':<40}")
print(f"{'-'*25} {'-'*6}  {'-'*40} {'-'*40}")

for r in all_results[:40]:  # Show first 40 results
    img_name = r['image'][:23] + '..' if len(r['image']) > 25 else r['image']
    p_text = r['paddle_text'][:38] + '..' if len(r['paddle_text']) > 40 else r['paddle_text']
    v_text = r['vietocr_text'][:38] + '..' if len(r['vietocr_text']) > 40 else r['vietocr_text']
    print(f"{img_name:<25} {r['paddle_conf']:6.3f}  {p_text:<40} {v_text:<40}")

if len(all_results) > 40:
    print(f"\n... and {len(all_results) - 40} more rows.")

---
## 7. Evaluation: PaddleOCR vs VietOCR

In [ ]:
def compute_cer(prediction, ground_truth):
    """
    Compute Character Error Rate using edit distance.
    CER = edit_distance(pred, gt) / len(gt)
    """
    pred = prediction.strip()
    gt = ground_truth.strip()
    if len(gt) == 0:
        return 0.0 if len(pred) == 0 else 1.0

    # Dynamic programming edit distance
    m, n = len(pred), len(gt)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if pred[i-1] == gt[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[m][n] / n


def compute_wer(prediction, ground_truth):
    """
    Compute Word Error Rate using edit distance on word tokens.
    """
    pred_words = prediction.strip().split()
    gt_words = ground_truth.strip().split()
    if len(gt_words) == 0:
        return 0.0 if len(pred_words) == 0 else 1.0

    m, n = len(pred_words), len(gt_words)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if pred_words[i-1] == gt_words[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[m][n] / n

In [ ]:
# Try to load ground truth labels for evaluation
# Attempt common label formats: paired txt files, CSV, or JSON

ground_truth = {}  # {image_filename: text}

# Strategy 1: Look for .txt files paired with images (same name, different ext)
for img_path in image_files[:BATCH_SIZE]:
    base = os.path.splitext(img_path)[0]
    for ext in ['.txt', '.gt.txt']:
        txt_path = base + ext
        if os.path.exists(txt_path):
            try:
                with open(txt_path, 'r', encoding='utf-8') as f:
                    ground_truth[os.path.basename(img_path)] = f.read().strip()
            except Exception:
                pass

# Strategy 2: Look for a labels CSV or JSON file
if not ground_truth:
    for lf in label_files:
        if lf.endswith('.csv'):
            try:
                import csv
                with open(lf, 'r', encoding='utf-8') as f:
                    reader = csv.reader(f)
                    header = next(reader, None)
                    for row in reader:
                        if len(row) >= 2:
                            ground_truth[row[0]] = row[1]
                if ground_truth:
                    print(f"Loaded labels from {os.path.basename(lf)}")
                    break
            except Exception:
                pass
        elif lf.endswith('.json'):
            try:
                with open(lf, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                if isinstance(data, dict):
                    ground_truth = data
                elif isinstance(data, list):
                    for item in data:
                        if 'image' in item and 'text' in item:
                            ground_truth[item['image']] = item['text']
                if ground_truth:
                    print(f"Loaded labels from {os.path.basename(lf)}")
                    break
            except Exception:
                pass

if ground_truth:
    print(f"\nFound {len(ground_truth)} ground truth labels.")
    print("\nSample labels:")
    for k, v in list(ground_truth.items())[:5]:
        print(f"  {k}: {v[:60]}")
else:
    print("No ground truth labels found. Will perform qualitative comparison only.")

In [ ]:
if ground_truth:
    # Quantitative evaluation against ground truth
    paddle_cers, paddle_wers = [], []
    vietocr_cers, vietocr_wers = [], []

    for r in all_results:
        img_name = r['image']
        if img_name in ground_truth:
            gt = ground_truth[img_name]
            paddle_cers.append(compute_cer(r['paddle_text'], gt))
            paddle_wers.append(compute_wer(r['paddle_text'], gt))
            vietocr_cers.append(compute_cer(r['vietocr_text'], gt))
            vietocr_wers.append(compute_wer(r['vietocr_text'], gt))

    if paddle_cers:
        print("Evaluation Results (lower is better):")
        print(f"{'Metric':<20} {'PaddleOCR':>12} {'VietOCR':>12}")
        print(f"{'-'*20} {'-'*12} {'-'*12}")
        print(f"{'Mean CER':<20} {np.mean(paddle_cers):>11.4f} {np.mean(vietocr_cers):>11.4f}")
        print(f"{'Mean WER':<20} {np.mean(paddle_wers):>11.4f} {np.mean(vietocr_wers):>11.4f}")
        print(f"{'Median CER':<20} {np.median(paddle_cers):>11.4f} {np.median(vietocr_cers):>11.4f}")
        print(f"{'Median WER':<20} {np.median(paddle_wers):>11.4f} {np.median(vietocr_wers):>11.4f}")
        print(f"{'Samples evaluated':<20} {len(paddle_cers):>12}")

        # Visualization
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        ax1.hist(paddle_cers, bins=20, alpha=0.6, label='PaddleOCR', color='steelblue')
        ax1.hist(vietocr_cers, bins=20, alpha=0.6, label='VietOCR', color='coral')
        ax1.set_xlabel('CER'); ax1.set_ylabel('Count'); ax1.set_title('Character Error Rate Distribution')
        ax1.legend()

        ax2.hist(paddle_wers, bins=20, alpha=0.6, label='PaddleOCR', color='steelblue')
        ax2.hist(vietocr_wers, bins=20, alpha=0.6, label='VietOCR', color='coral')
        ax2.set_xlabel('WER'); ax2.set_ylabel('Count'); ax2.set_title('Word Error Rate Distribution')
        ax2.legend()

        plt.tight_layout()
        plt.show()
    else:
        print("No matching labels found for processed images.")

else:
    # Qualitative comparison: PaddleOCR vs VietOCR
    print("Qualitative Comparison: PaddleOCR vs VietOCR")
    print("(No ground truth available, showing side-by-side predictions)\n")

    # Identify where the two models disagree (most interesting cases)
    disagreements = [r for r in all_results
                     if r['paddle_text'].strip() != r['vietocr_text'].strip()
                     and r['vietocr_text'] not in ('[error]', '[crop failed]', '')]
    agreements = [r for r in all_results
                  if r['paddle_text'].strip() == r['vietocr_text'].strip()]

    print(f"Total text regions: {len(all_results)}")
    print(f"Models agree: {len(agreements)} ({100*len(agreements)/max(len(all_results),1):.1f}%)")
    print(f"Models disagree: {len(disagreements)} ({100*len(disagreements)/max(len(all_results),1):.1f}%)")

    if disagreements:
        print(f"\nTop disagreements (likely diacritic differences):")
        print(f"{'Image':<25} {'PaddleOCR':<35} {'VietOCR':<35}")
        print(f"{'-'*25} {'-'*35} {'-'*35}")
        for r in disagreements[:15]:
            img = r['image'][:23] + '..' if len(r['image']) > 25 else r['image']
            p = r['paddle_text'][:33] + '..' if len(r['paddle_text']) > 35 else r['paddle_text']
            v = r['vietocr_text'][:33] + '..' if len(r['vietocr_text']) > 35 else r['vietocr_text']
            print(f"{img:<25} {p:<35} {v:<35}")

---
## 8. Practical Application: Document OCR Pipeline

In [ ]:
def document_ocr(image_path, use_vietocr=True, visualize=True):
    """
    Complete OCR pipeline for a single document image.

    Pipeline:
        1. PaddleOCR detects text regions (bounding boxes)
        2. Regions are sorted top-to-bottom, left-to-right
        3. VietOCR (optionally) re-recognizes each region
        4. Returns structured output with all text

    Args:
        image_path: path to input image
        use_vietocr: if True, use VietOCR for recognition (better Vietnamese)
        visualize: if True, display annotated image

    Returns:
        dict with keys: 'image', 'lines' (list of dicts), 'full_text'
    """
    # Step 1: Detection
    detections = run_paddleocr(image_path)

    # Step 2: Sort by vertical position (top to bottom), then left to right
    def sort_key(det):
        bbox = det[0]
        y_center = np.mean([pt[1] for pt in bbox])
        x_center = np.mean([pt[0] for pt in bbox])
        return (y_center, x_center)

    detections.sort(key=sort_key)

    # Step 3: Recognition
    lines = []
    for bbox, paddle_text, conf in detections:
        text = paddle_text
        source = 'paddleocr'

        if use_vietocr:
            crop = crop_text_region(image_path, bbox)
            if crop is not None:
                try:
                    text = vietocr_predictor.predict(crop)
                    source = 'vietocr'
                except Exception:
                    pass  # Fall back to PaddleOCR text

        lines.append({
            'text': text,
            'confidence': round(conf, 4),
            'source': source,
            'bbox': bbox
        })

    # Step 4: Assemble full text
    full_text = '\n'.join(line['text'] for line in lines)

    result = {
        'image': os.path.basename(image_path),
        'num_lines': len(lines),
        'lines': lines,
        'full_text': full_text
    }

    # Visualization
    if visualize:
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 10),
                                        gridspec_kw={'width_ratios': [1, 1]})

        ax1.imshow(img)
        for idx, line in enumerate(lines):
            pts = np.array(line['bbox'])
            polygon = patches.Polygon(pts, linewidth=1.5,
                                      edgecolor='lime', facecolor='lime', alpha=0.15)
            ax1.add_patch(polygon)
            ax1.text(pts[0][0], pts[0][1] - 3, str(idx+1),
                     fontsize=7, color='white',
                     bbox=dict(boxstyle='round,pad=0.15', facecolor='green', alpha=0.7))
        ax1.set_title('Detected Text Regions', fontsize=12)
        ax1.axis('off')

        # Extracted text
        ax2.axis('off')
        display_text = f"Extracted Text ({len(lines)} lines):\n{'='*45}\n\n"
        for idx, line in enumerate(lines):
            display_text += f"{idx+1:2d}. {line['text']}\n"
        ax2.text(0.02, 0.98, display_text, transform=ax2.transAxes,
                 fontsize=9, verticalalignment='top', fontfamily='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
        ax2.set_title('Recognized Text (VietOCR)' if use_vietocr else 'Recognized Text (PaddleOCR)',
                      fontsize=12)

        plt.suptitle(f'Document OCR: {os.path.basename(image_path)}', fontsize=14)
        plt.tight_layout()
        plt.show()

    return result

In [ ]:
# Demo: Run the full document OCR pipeline on sample images
demo_images = image_files[:3] if len(image_files) >= 3 else image_files

for img_path in demo_images:
    print(f"\n{'#'*70}")
    print(f"# Document OCR: {os.path.basename(img_path)}")
    print(f"{'#'*70}")

    result = document_ocr(img_path, use_vietocr=True, visualize=True)

    print(f"\nFull extracted text:")
    print(f"{'-'*50}")
    print(result['full_text'])
    print(f"{'-'*50}")
    print(f"Lines detected: {result['num_lines']}")

---
## 9. Summary & Insights

### Model Comparison

| Aspect | PaddleOCR | VietOCR |
|--------|-----------|--------|
| **Pipeline** | End-to-end (detect + recognize) | Recognition only |
| **Vietnamese support** | Good (built-in `lang='vi'`) | Excellent (purpose-built) |
| **Diacritics accuracy** | Decent, sometimes misses tonal marks | Superior handling of dấu |
| **Speed** | Fast (single pass) | Slower (requires crops from detector) |
| **Best use case** | Quick full-pipeline OCR | High-accuracy Vietnamese text |

### Recommended Approach

Use **PaddleOCR for detection** (bounding boxes) + **VietOCR for recognition** (text). This hybrid approach leverages the strengths of both models:
- PaddleOCR's robust text detection handles diverse layouts
- VietOCR's Vietnamese-optimized recognition produces more accurate diacritics

### Key Challenges Observed

1. **Handwritten text**: Both models struggle with handwriting; fine-tuning needed
2. **Low-resolution images**: Small or blurry text degrades recognition
3. **Complex layouts**: Tables, multi-column text, overlapping regions
4. **Mixed languages**: Documents with Vietnamese + English/numbers

### Next Steps

- **Fine-tune VietOCR** on domain-specific data (e.g., ID cards, receipts, medical forms)
- **Post-processing**: Vietnamese spell-checking and language model correction
- **Layout analysis**: Detect tables, headers, paragraphs for structured extraction
- **Augmentation**: Train with synthetic Vietnamese text to improve robustness